# Record CLOB order book updates

In notebooks, prefer the async methods (`await recorder.aconnect()`, `await recorder.arecord(...)`).

This notebook shows two patterns:

1) Resolve `conditionId` from a Polymarket URL, then call `asubscribe(condition_id)`.
2) Call `asubscribe_url(url, ...)` directly.


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "clients").exists() else cwd.parent
sys.path.insert(0, str(REPO_ROOT))
REPO_ROOT


In [ ]:
from clients.gamma_client import GammaClient
from collectors.orderbook_recorder import OrderBookRecorder

gamma = GammaClient()
recorder = OrderBookRecorder(gamma_client=gamma)

url = "https://polymarket.com/event/fed-decision-in-march-885"
markets = gamma.resolve_markets_from_polymarket_url(url)
[(i, m.get("slug"), m.get("conditionId")) for i, m in enumerate(markets)]


In [ ]:
await recorder.aconnect()

market_index = 0  # change if the event has multiple markets
m0 = markets[market_index]
condition_id = m0.get("conditionId") or m0.get("condition_id")

# Option A: resolve condition id yourself
# token_ids = await recorder.asubscribe(condition_id)

# Option B: subscribe directly from URL (equivalent)
token_ids = await recorder.asubscribe_url(url, market_index=market_index)

token_ids


In [ ]:
snapshot = await recorder.aget_snapshot()
df = await recorder.arecord(10, max_messages=500)

out_path = REPO_ROOT / "data" / "orderbook_messages.parquet"
recorder.save_to_parquet(df, out_path)
await recorder.aclose()

df.head()
